In [1]:
!pip install belt_nlp
!pip install datasets
!pip install accelerate -U
!pip install transformers[torch] -U

  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.19.3-py3-none-manylinux1_x86_64.whl (166.0 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (99 kB)
  Using cached nvidia_nvjitlink_cu12-12.4.127-py3-none-m

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from typing import Optional

import torch
from torch import Tensor
from transformers import BatchEncoding, PreTrainedTokenizerBase
from transformers import AutoTokenizer
from transformers import Trainer, AutoModelForSeq2SeqLM
from transformers import TrainingArguments


from belt_nlp.exceptions import InconsistentSplittingParamsException

In [4]:
def check_split_parameters_consistency(chunk_size: int, stride: int, minimal_chunk_length: int) -> None:
    if chunk_size > 510:
        raise InconsistentSplittingParamsException("Size of each chunk cannot be bigger than 510!")
    if minimal_chunk_length > chunk_size:
        raise InconsistentSplittingParamsException("Minimal length cannot be bigger than size!")
    if stride > chunk_size:
        raise InconsistentSplittingParamsException(
            "Stride cannot be bigger than size! Chunks must overlap or be near each other!"
        )

In [5]:
def split_tokens_into_smaller_chunks(
    tokens: BatchEncoding,
    chunk_size: int,
    stride: int,
    minimal_chunk_length: int,
) -> tuple[list[Tensor], list[Tensor]]:
    """Splits tokens into overlapping chunks with given size and stride."""
    input_id_chunks = split_overlapping(tokens["input_ids"][0], chunk_size, stride, minimal_chunk_length)
    mask_chunks = split_overlapping(tokens["attention_mask"][0], chunk_size, stride, minimal_chunk_length)
    return input_id_chunks, mask_chunks


In [6]:
def split_overlapping(tensor: Tensor, chunk_size: int, stride: int, minimal_chunk_length: int) -> list[Tensor]:
    """Helper function for dividing 1-dimensional tensors into overlapping chunks."""
    check_split_parameters_consistency(chunk_size, stride, minimal_chunk_length)
    result = [tensor[i : i + chunk_size] for i in range(0, len(tensor), stride)]
    if len(result) > 1:
        # ignore chunks with less than minimal_length number of tokens
        result = [x for x in result if len(x) >= minimal_chunk_length]
    return result

In [7]:
def add_special_tokens_at_beginning_and_end(input_id_chunks: list[Tensor], mask_chunks: list[Tensor]) -> None:
    """
    Adds special CLS token (token id = 101) at the beginning.
    Adds SEP token (token id = 102) at the end of each chunk.
    Adds corresponding attention masks equal to 1 (attention mask is boolean).
    """
    for i in range(len(input_id_chunks)):
        # adding CLS (token id 101) and SEP (token id 102) tokens
        input_id_chunks[i] = torch.cat([Tensor([101]), input_id_chunks[i], Tensor([102])])
        # adding attention masks  corresponding to special tokens
        mask_chunks[i] = torch.cat([Tensor([1]), mask_chunks[i], Tensor([1])])

In [8]:
def add_padding_tokens(input_id_chunks: list[Tensor], mask_chunks: list[Tensor]) -> None:
    """Adds padding tokens (token id = 0) at the end to make sure that all chunks have exactly 512 tokens."""
    for i in range(len(input_id_chunks)):
        # get required padding length
        pad_len = 512 - input_id_chunks[i].shape[0]
        # check if tensor length satisfies required chunk size
        if pad_len > 0:
            # if padding length is more than 0, we must add padding
            input_id_chunks[i] = torch.cat([input_id_chunks[i], Tensor([0] * pad_len)])
            mask_chunks[i] = torch.cat([mask_chunks[i], Tensor([0] * pad_len)])

In [9]:
def stack_tokens_from_all_chunks(input_id_chunks: list[Tensor], mask_chunks: list[Tensor]) -> tuple[Tensor, Tensor]:
    """Reshapes data to a form compatible with BERT model input."""
    input_ids = torch.stack(input_id_chunks)
    attention_mask = torch.stack(mask_chunks)

    return input_ids.long(), attention_mask.int()


In [10]:
def tokenize_whole_text(text: str, tokenizer: PreTrainedTokenizerBase) -> BatchEncoding:
    """Tokenizes the entire text without truncation and without special tokens."""
    tokens = tokenizer(text, add_special_tokens=False, truncation=False, return_tensors="pt")
    return tokens

In [11]:
def tokenize_text_with_truncation(
    text: str, tokenizer: PreTrainedTokenizerBase, maximal_text_length: int
) -> BatchEncoding:
    """Tokenizes the text with truncation to maximal_text_length and without special tokens."""
    tokens = tokenizer(
        text, add_special_tokens=False, max_length=maximal_text_length, truncation=True, return_tensors="pt"
    )
    return tokens

In [12]:
def equalize_padding(input_id_chunks, mask_chunks, pad_token_id=0):
    max_length = max(chunk.shape[1] for chunk in input_id_chunks)  # Find the maximum length across all chunks
    padded_input_ids = []
    padded_masks = []

    for input_ids, mask in zip(input_id_chunks, mask_chunks):
        # Calculate how much padding is needed
        padding_length = max_length - input_ids.shape[1]

        # Pad the input_ids and attention masks
        padded_input_ids.append(torch.cat([
            input_ids,
            torch.full((input_ids.shape[0], padding_length), pad_token_id, dtype=torch.long)
        ], dim=1))

        padded_masks.append(torch.cat([
            mask,
            torch.zeros((mask.shape[0], padding_length), dtype=torch.long)
        ], dim=1))

    return padded_input_ids, padded_masks

# # Use this function right before stacking
# padded_input_id_chunks, padded_mask_chunks = equalize_padding(input_id_chunks, mask_chunks)
# input_ids = torch.stack(padded_input_id_chunks)
# attention_mask = torch.stack(padded_mask_chunks)


In [13]:
def transform_single_text(
    text: str,
    tokenizer: PreTrainedTokenizerBase,
    chunk_size: int,
    stride: int,
    minimal_chunk_length: int,
    maximal_text_length: Optional[int],
) -> tuple[Tensor, Tensor]:
    """Transforms (the entire) text to model input of BERT model."""
    if maximal_text_length:
        tokens = tokenize_text_with_truncation(text, tokenizer, maximal_text_length)
    else:
        tokens = tokenize_whole_text(text, tokenizer)
    input_id_chunks, mask_chunks = split_tokens_into_smaller_chunks(tokens, chunk_size, stride, minimal_chunk_length)
    add_special_tokens_at_beginning_and_end(input_id_chunks, mask_chunks)
    add_padding_tokens(input_id_chunks, mask_chunks)
    #padded_input_id_chunks, padded_mask_chunks = equalize_padding(input_id_chunks, mask_chunks)
    #input_ids, attention_mask = stack_tokens_from_all_chunks(padded_input_id_chunks, padded_mask_chunks)
    input_ids, attention_mask = stack_tokens_from_all_chunks(input_id_chunks, mask_chunks)
    return input_ids, attention_mask

In [14]:
def transform_list_of_texts(
    texts: list[str],
    tokenizer: PreTrainedTokenizerBase,
    chunk_size: int,
    stride: int,
    minimal_chunk_length: int,
    maximal_text_length: Optional[int] = None,
) -> BatchEncoding:
    model_inputs = [
        transform_single_text(text, tokenizer, chunk_size, stride, minimal_chunk_length, maximal_text_length)
        for text in texts
    ]
    input_ids = [model_input[0] for model_input in model_inputs]
    attention_mask = [model_input[1] for model_input in model_inputs]
    tokens = {"input_ids": input_ids, "attention_mask": attention_mask}
    return BatchEncoding(tokens)

In [15]:
source_file_path = '/content/drive/MyDrive/codeT5p_data_@/train_data/train.src'
target_file_path = '/content/drive/MyDrive/codeT5p_data_@/train_data/train.tgt'

def load_data(input_path, target_path):
    with open(input_path, 'r', encoding='utf-8') as file:
        inputs = [line.strip() for line in file.readlines()]

    with open(target_path, 'r', encoding='utf-8') as file:
        targets = [line.strip() for line in file.readlines()]

    return inputs, targets
inputs, targets = load_data(source_file_path, target_file_path)


In [16]:
len(inputs)

1375

In [17]:
pretrained_model_codet5p = "Salesforce/codet5p-220m"
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_codet5p)

chunk_size = 500 # Chunk size for inputs
stride = 250     # Overlap size for input chunks
minimal_chunk_length = 100

# Tokenize and chunk input texts
tokenized_inputs = transform_list_of_texts(inputs, tokenizer, chunk_size, stride, minimal_chunk_length)

# Tokenized the targets without need of chunks
tokenized_targets = [tokenizer(tgt, add_special_tokens=True, padding="max_length", max_length=512, truncation=True)['input_ids'] for tgt in targets]


/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/703k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/294k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/12.5k [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (6420 > 512). Running this sequence through the model will result in indexing errors


In [18]:
# Assuming tokenized_inputs contains keys like 'input_ids' and 'attention_mask', each being a list of lists
# Check the total number of chunks
num_input_chunks = len(tokenized_inputs['input_ids'])
print(f"Number of chunks in tokenized inputs: {num_input_chunks}")

# Since each entry in `tokenized_inputs['input_ids']` is a list (a chunk), you might want to know total inputs:
total_input_length = sum(len(chunk) for chunk in tokenized_inputs['input_ids'])
print(f"Total length of all input chunks combined: {total_input_length}")

# Checking the length of tokenized targets
num_targets = len(tokenized_targets)
print(f"Number of tokenized targets: {num_targets}")

# Optionally, you can check details about the size of each target:
target_lengths = [len(t) for t in tokenized_targets]
print(f"Tokenized target lengths: {target_lengths}")


Number of chunks in tokenized inputs: 1375
Total length of all input chunks combined: 45286
Number of tokenized targets: 1375
Tokenized target lengths: [512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 512, 51

In [84]:
# To check if any chunks are empty
# Assuming tokenized_inputs is a dictionary containing 'input_ids' and 'attention_mask'
# which are lists of lists or similar structures containing token ids.

empty_chunks_count = 0

for chunk in tokenized_inputs['attention_mask']:
    # Check if the chunk is empty or has no tokens
    if len(chunk) == 0:
        empty_chunks_count += 1

print(f"Number of empty chunks: {empty_chunks_count}")


Number of empty chunks: 0


In [19]:
# from torch.utils.data import Dataset

# class Seq2SeqDataset(Dataset):
#     def __init__(self, tokenized_inputs: BatchEncoding, tokenized_targets: Tensor):
#         self.tokenized_inputs = tokenized_inputs
#         self.tokenized_targets = tokenized_targets

#     def __len__(self):
#         return len(self.tokenized_inputs["input_ids"])

#     def __getitem__(self, idx):
#         input_ids = self.tokenized_inputs["input_ids"][idx]
#         attention_mask = self.tokenized_inputs["attention_mask"][idx]
#         target_ids = self.tokenized_targets[idx]
#         return {
#             "input_ids": input_ids,
#             "attention_mask": attention_mask,
#             "labels": target_ids
#         }

# # Create dataset
# dataset = Seq2SeqDataset(tokenized_inputs, tokenized_targets)


In [20]:
# from torch.utils.data import DataLoader

# def create_data_loader(dataset, batch_size=4, shuffle=True):
#     loader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=4)
#     return loader

# train_loader = create_data_loader(dataset)


/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


In [26]:
# from transformers import T5ForConditionalGeneration, AdamW

# # Load pre-trained model
# model = T5ForConditionalGeneration.from_pretrained(pretrained_model_codet5p)

# # Prepare optimizer
# optimizer = AdamW(model.parameters(), lr=5e-5)

# # Training loop
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)
# num_epochs = 5
# for epoch in range(num_epochs):
#     model.train()
#     for batch in train_loader:
#         batch = {k: v.to(device) for k, v in batch.items()}
#         outputs = model(**batch)
#         loss = outputs.loss
#         loss.backward()
#         optimizer.step()
#         optimizer.zero_grad()
#         print(f"Loss: {loss.item()}")

# # Optionally save the model
# model.save_pretrained("/path/to/save/model")


In [28]:
# Detailed inspection of tokenized_inputs
print("Detailed inspection of tokenized_inputs:")
print(type(tokenized_inputs))  # Should print BatchEncoding
print(tokenized_inputs.keys())  # Should show keys like 'input_ids', 'attention_mask'

# Check the structure of the first few entries in the 'input_ids' list
print("Number of chunks:", len(tokenized_inputs['input_ids']))
print("First few chunks' details:")
for i in range(min(5, len(tokenized_inputs['input_ids']))):  # Only inspect the first 5 chunks
    chunk_input_ids = tokenized_inputs['input_ids'][i]
    chunk_attention_mask = tokenized_inputs['attention_mask'][i]
    print(f"Chunk {i+1}:")
    print(" Length of Input IDs:", len(chunk_input_ids))  # Shows the length of this particular chunk
    print(" Sample Input IDs:", chunk_input_ids[:5])  # Show first 10 tokens of this chunk


Detailed inspection of tokenized_inputs:
<class 'transformers.tokenization_utils_base.BatchEncoding'>
dict_keys(['input_ids', 'attention_mask'])
Number of chunks: 1375
First few chunks' details:
Chunk 1:
 Length of Input IDs: 26
 Sample Input IDs: tensor([[  101,  2978,  1241,  ...,     0,     0,     0],
        [  101,   269,   876,  ...,     0,     0,     0],
        [  101,    67,  2556,  ...,     0,     0,     0],
        [  101, 14295,    67,  ...,     0,     0,     0],
        [  101,  3631,   269,  ...,     0,     0,     0]])
Chunk 2:
 Length of Input IDs: 1
 Sample Input IDs: tensor([[  101,  2978,  1241,    67,  1153,   261,   269,   513,    67,  1937,
            16,   269,   513,    67,  8734,    16,   269,   513,    67, 20390,
            16,   269,   513,    67,  1672,    16,   269, 13999,    16,   269,
           513,    67,  2463,   269, 11272,   269,   810,   513,    67,  1937,
            31,   269,   876,   513,    67,  8734,    31,   269,   876,   513,
            67

In [54]:
# from transformers import AutoTokenizer

# checkpoint = "Salesforce/codet5p-220m"
# tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

In [27]:
# for batch in loader:
#     print("Input IDs:", batch['input_ids'].shape)
#     print("Attention Mask:", batch['attention_mask'].shape)
#     print("Labels:", batch['labels'].shape)
#     break  # Show only the first batch

In [22]:
from torch.utils.data import Dataset, DataLoader

class Seq2SeqDataset(Dataset):
    def __init__(self, tokenized_inputs, tokenized_targets):
        self.input_ids = tokenized_inputs['input_ids']
        self.attention_masks = tokenized_inputs['attention_mask']
        self.targets = tokenized_targets  # Assuming this is already a list of lists (one per target sequence)

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        input_ids = self.input_ids[idx]
        attention_mask = self.attention_masks[idx]
        target_ids = self.targets[idx]

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': target_ids  # In Hugging Face transformers, targets are usually labeled as 'labels'
        }


In [28]:
from torch.nn.utils.rnn import pad_sequence

def collate_batch(batch):
    input_ids = [item['input_ids'] for item in batch]
    attention_masks = [item['attention_mask'] for item in batch]
    labels = [item['labels'] for item in batch]

    # Pad sequences so they are the same length
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_masks = pad_sequence(attention_masks, batch_first=True, padding_value=0)
    labels = pad_sequence(labels, batch_first=True, padding_value=tokenizer.pad_token_id)  # Assuming pad_token_id is appropriate for labels too

    return {
        'input_ids': input_ids,
        'attention_mask': attention_masks,
        'labels': labels
    }


In [29]:
# Initialize your dataset
dataset = Seq2SeqDataset(tokenized_inputs, tokenized_targets)

# Create a DataLoader
batch_size = 4  # Set batch size as required
#loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)


In [30]:
from transformers import AutoModelForSeq2SeqLM, AdamW

# Load the pre-trained model
model = AutoModelForSeq2SeqLM.from_pretrained(pretrained_model_codet5p)

# Move model to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

# Prepare optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)


In [31]:
from torch.nn.utils import clip_grad_norm_

epochs = 3  # Set number of epochs as needed
model.train()

for epoch in range(epochs):
    for batch in loader:
        optimizer.zero_grad()

        # Load batch to the appropriate device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backward pass
        loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        print(f"Epoch: {epoch}, Loss: {loss.item()}")


TypeError: expected Tensor as element 0 in argument 0, but got list

In [ ]:
model.save_pretrained('./my_finetuned_codet5p')
tokenizer.save_pretrained('./my_finetuned_codet5p')
